# 06 — Vérifier, publier et conserver les résultats

**Objectif :** comprendre les preuves nécessaires avant l'envoi et retrouver ensuite exactement les poids, le code et les images utilisés.

~~~text
Run terminé → revue complète → export → loader exporté sur GPU libre
            → 2 vraies générations RGBA → collecte des petits fichiers
            → publication HF et contrôle de la révision immuable
            → archive de résultats dans Git → commit Angular et push main
~~~

La publication vise le dépôt Base autorisé et conserve sa visibilité. L'authentification reste dans le gestionnaire HF local Windows : aucun jeton ou mot de passe dans les cellules, les sorties ou Git.

In [ ]:
from pathlib import Path
import sys

book_dir = next(
    folder
    for parent in (Path.cwd(), *Path.cwd().parents)
    for folder in (parent / "notebooks", parent / "2-LoRA-qwen/notebooks", parent)
    if (folder / "notebook_utils.py").is_file()
)
if str(book_dir) not in sys.path:
    sys.path.insert(0, str(book_dir))
from notebook_utils import (
    PROJECT, REPO, ARTIFACTS, read_json, read_jsonl, sha256,
    contained, current_dataset, archived_report, result_file,
    show_json, table, show_image, run_script,
)
print("Projet :", PROJECT.name)


In [ ]:
show_json(result_file("publication-status.json"))
verification = read_json(result_file("release/export-validation.json"))
if verification:
    table([{"contrôle": key, "valeur": verification.get(key)} for key in (
        "passed", "projector_roundtrip_exact", "loader_from_release",
        "projector_sha256", "release_manifest_sha256", "scope"
    )], ["contrôle", "valeur"])
else:
    print("Le loader exporté n'a pas encore de rapport disponible ici.")

## Les deux générations du loader livré

Le vérificateur importe le code de l'export, recharge le projecteur et utilise le vrai encodeur FP8. Les deux cas sont un prompt naturel et un prompt avec boîtes. Les PNG doivent contenir un sujet et de la transparence. Ces contrôles sont techniques ; ils ne remplacent pas la revue spatiale.

In [ ]:
if verification:
    for method in ("candidate_text", "candidate_layout"):
        image = result_file(f"release/validation/{method}.png")
        if image.exists():
            expected = verification["images"].get(f"validation/{method}.png")
            if sha256(image) != expected:
                raise ValueError("Image de vérification modifiée.")
            show_image(image, method)

## Préparation et vérification sur le serveur

Ces actions demandent un run terminé et réellement revu. Le vérificateur prend le verrou du run et refuse une chaîne incomplète. Les drapeaux restent à False dans le notebook versionné.

In [ ]:
RUN_PREPARE_EXPORT = False
RUN_VERIFY_EXPORTED_LOADER = False
run_script("publish/publish_spatial.py", ["prepare"], enabled=RUN_PREPARE_EXPORT)
run_script("publish/verify_spatial_export.py",
           enabled=RUN_VERIFY_EXPORTED_LOADER, gpu=True)

## Collecte et envoi depuis Windows

Rapatrier les petits résultats et les nouveaux poids du projecteur ; réutiliser les gros poids publics locaux identiques par hardlinks après contrôle SHA. La collecte ne lance pas d'inférence. Le publisher vérifie le compte, la visibilité, la révision de départ et les empreintes distantes après envoi.

La procédure SSH/rsync complète est documentée dans [publish/README.md](../publish/README.md). Les adresses et identifiants de la machine restent dans les instructions locales hors Git.

In [ ]:
RUN_COLLECT_RELEASE = False
RUN_HF_PUSH = False
if (RUN_COLLECT_RELEASE or RUN_HF_PUSH) and sys.platform != "win32":
    raise RuntimeError("Exécuter la collecte et le push avec le Python Windows et son authentification HF canonique.")
run_script("publish/collect_remote_release.py", enabled=RUN_COLLECT_RELEASE)
run_script("publish/publish_spatial.py", ["push"], enabled=RUN_HF_PUSH)

## Archiver les résultats pour Git

Après publication vérifiée, l'archive conserve les historiques, la galerie complète, la revue identifiée, les deux inférences de vérification, les paramètres, empreintes et révisions. Elle exclut les poids, caches, optimiseur, journaux et coordonnées de connexion.

L'archive ne réécrit pas les PNG ni leurs manifestes et ne se remplace pas silencieusement. Le pointeur `reports/current.json` sélectionne le rapport lu par les notebooks. Le script contrôle la cohérence des preuves avant de créer cette archive.

In [ ]:
RUN_ARCHIVE_FINAL_RESULTS = False
run_script("notebooks/archive_spatial_results.py", enabled=RUN_ARCHIVE_FINAL_RESULTS)
archive = archived_report()
if archive:
    show_json(archive / "summary.json")
    print("Manifeste de l'archive :", sha256(archive / "manifest.json"))
else:
    print("Archivage final en attente de résultats réellement terminés et publiés.")

## Versionnement et lecture future

1. Nettoyer les sorties des notebooks avec `python 2-LoRA-qwen/notebooks/clean_notebooks.py`.
2. Examiner les modifications, les fichiers sélectionnés et les empreintes des rapports.
3. Créer un commit Angular, par exemple `docs(notebooks): archive the verified spatial training results`.
4. Pousser sur `origin/main`, sans force push, puis vérifier la révision distante.

Le code, les notebooks, les PNG sources, le dataset validé et les rapports finaux sont dans Git. Les poids sont sur HF à une révision immuable. Les secrets, environnements, caches, journaux et brouillons restent hors Git.

Voir [reports/README.md](../reports/README.md) pour le format de conservation. Revenir au [sommaire](README.md).